In [5]:
import numpy as np
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load in the data
data_url = '../spambase_data/spambase.data'
names_url = '../spambase_data/spambase.names'

# parse the feature names from the .names file
feature_names = []
with open(names_url, "r") as f:
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith("|"):
            feature_names.append(line.split(":")[0].strip())
feature_names.append("label") # last column is the class label

df = pd.read_csv(data_url, header=None, names=feature_names)
x = df.drop("label", axis=1).values
y = df["label"].values
names = feature_names[:-1]

# Create the train test split for the data
x_tr, x_te, y_tr, y_te = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y 
)

# Train the logistic regrerssion model
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)

# The model will be used at the end for comparison
model = LogisticRegression(max_iter=2000, solver="lbfgs", C=1.0, random_state=42)
model.fit(x_tr, y_tr)

# sigmoid and cross-entropy loss
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy_loss(x, y, w, b):
    n = len(y)
    p = sigmoid(x @ w + b)
    # clip to avoid log(0)
    p = np.clip(p, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1-p))

# Gradient descent for logistic regression
def logistic_gradient_descent(x, y, lr, n_iter, report_at={10, 50, 100}):
    n, d = x.shape
    w = np.zeros(d)
    b = 0.0
    losses = {}

    for t in range(1, n_iter + 1):
        p = sigmoid(x @ w + b)
        error = p - y
        grad_w = x.T @ error / n
        grad_b = np.mean(error)

        w -= lr * grad_w
        b -= lr * grad_b

        if t in report_at:
            losses[t] = cross_entropy_loss(x, y, w, b)

    return w, b, losses

# predict with learned weights
def predict(x, w, b, threshold=0.5):
    return(sigmoid(x @ w + b) >= threshold).astype(int)

# Metrics
def metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_pred, y_true, zero_division=0)
    return acc, prec, rec, f1

learning_rates = [0.01, 0.1, 1.0]
n_iter = 100
report_at = {10, 50, 100}

print("=" * 60)
print(" Part 2 - Gradient descent logistic regression")
print("=" * 60)

results = {}
for lr in learning_rates:
    w, b, losses = logistic_gradient_descent(
        x_tr, y_tr, lr=lr, n_iter=n_iter, report_at=report_at)
    results[lr] = (w, b, losses)

    print(f"\n-- Learning Rate: {lr} --")
    print(f"    {'Iteration':>12} {'Cross-entropy Loss':>20}")
    print(f"    {'-'*34}")
    for it in sorted(losses):
        print(f"    {it:<12} {losses[it]:>20.6f}")

# Metricts at 100 iterations
print(f"\n{'='*60}")
print(" Metrics at 100 Iterations (Test set)")
print(f"{'='*60}")
print(f"\n{'LR':<6} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 50)

for lr in learning_rates:
    w, b, _ = results[lr]
    y_pred_te = predict(x_te, w, b)
    acc, prec, rec, f1 = metrics(y_te, y_pred_te)
    print(f"{lr:<6} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

# compare with sklearn
print(f"\n {'='*60}")
print(" Comparison: GD vs sklearn Logistic Regression")
print(f"{'='*60}")

y_pred_sk_tr = model.predict(x_tr)
acc, prec, rec, f1 = metrics(y_tr, y_pred_sk_tr)
print(f"\nsklearn    (train): Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")

y_pred_sk_te = model.predict(x_te)
acc, prec, rec, f1 = metrics(y_te, y_pred_sk_te) 
print(f"\nsklearn    (test): Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")

best_lr = 0.1
w, b, _ = results[best_lr]
y_pred_gr_tr = predict(x_tr, w, b)
acc, prec, rec, f1 = metrics(y_tr, y_pred_gr_tr)
print(f"\nGD lr={best_lr} (train): Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")

y_pred_gr_te = predict(x_te, w, b)
acc, prec, rec, f1 = metrics(y_te, y_pred_gr_te)
print(f"\nGD lr={best_lr} (test): Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")

 Part 2 - Gradient descent logistic regression

-- Learning Rate: 0.01 --
       Iteration   Cross-entropy Loss
    ----------------------------------
    10                       0.651338
    50                       0.542090
    100                      0.469330

-- Learning Rate: 0.1 --
       Iteration   Cross-entropy Loss
    ----------------------------------
    10                       0.465983
    50                       0.325163
    100                      0.289826

-- Learning Rate: 1.0 --
       Iteration   Cross-entropy Loss
    ----------------------------------
    10                       0.285940
    50                       0.243037
    100                      0.230871

 Metrics at 100 Iterations (Test set)

LR       Accuracy  Precision     Recall         F1
--------------------------------------------------
0.01       0.8966     0.8731     0.8634     0.8682
0.1        0.9036     0.9054     0.8436     0.8734
1.0        0.9218     0.9252     0.8722     0.8980

 Comp